# Capstone — Ranking Signal Analysis

This capstone consolidates the completed ML-02 through ML-07 work. It reports the March 2026 warehouse slice, five decision-time features, the leakage check, and the transparent baseline queue. ML-08 model results are intentionally not invented; the notebook records that comparison as pending.

## 1. Question

Which observable search signals should an editor investigate first when prioritizing content review? The decision output is a ranked, human-readable queue. One source row is one pseudonymized content item for one client on one report date; the lane aggregates daily rows to content-level March review records. A wrong call costs review time or an unnecessary edit, so every recommendation remains decision-support rather than an automatic action.

In [8]:
import pandas as pd

question = {
    "lane": "Ranking Signal Analysis",
    "decision": "Which observable search signals should an editor investigate first?",
    "unit": "one pseudonymized content item aggregated from daily client/content observations",
    "output": "ranked queue with score, one reason code, and one action label",
    "claim_boundary": "observed, directional, decision-support; not causal",
}
for key, value in question.items():
    print(f"{key}: {value}")
assert question["lane"] == "Ranking Signal Analysis"
assert "causal" in question["claim_boundary"]

lane: Ranking Signal Analysis
decision: Which observable search signals should an editor investigate first?
unit: one pseudonymized content item aggregated from daily client/content observations
output: ranked queue with score, one reason code, and one action label
claim_boundary: observed, directional, decision-support; not causal


## 2. Data

The analysis uses the pseudonymized FlyRank warehouse `fact_content_daily_performance` March 2026 partition. Features are available through March 15; March 16–31 is reserved for the observed movement proxy. The grain check found no duplicate client/content/date keys. Identifiers remain grouping context only, and future-window, label-derived, product-decision, query, URL, and credential fields are excluded.

In [9]:
data_facts = {
    "release": "FlyRank internship-warehouse",
    "table": "fact_content_daily_performance",
    "partition": "month=2026-03",
    "daily_rows": 9_841_378,
    "distinct_content_items": 331_437,
    "duplicate_daily_grains": 0,
    "rows_with_ga4_available_true": 413_966,
    "feature_window": "2026-03-01 through 2026-03-15",
    "outcome_window": "2026-03-16 through 2026-03-31",
}
print(pd.DataFrame([data_facts]).to_string(index=False))
assert data_facts["duplicate_daily_grains"] == 0
assert data_facts["daily_rows"] > data_facts["distinct_content_items"]

                     release                          table     partition  daily_rows  distinct_content_items  duplicate_daily_grains  rows_with_ga4_available_true                feature_window                outcome_window
FlyRank internship-warehouse fact_content_daily_performance month=2026-03     9841378                  331437                       0                        413966 2026-03-01 through 2026-03-15 2026-03-16 through 2026-03-31


## 3. Methodology

The decision-time feature vector contains five first-half Search Console signals. The decline proxy is 1 when second-half impressions per day are below 80% of first-half impressions per day. The transparent baseline uses volume points plus a position-aware low-CTR screen. Validation is client-grouped so one client's repeated patterns do not appear in both train and test. Leakage checks deliberately add an exact label copy, verify the score jumps, then remove it.

In [10]:
feature_columns = [
    "first_half_impressions",
    "first_half_clicks",
    "first_half_ctr",
    "first_half_avg_position",
    "first_half_active_days",
]
method_facts = {
    "features": feature_columns,
    "label_proxy": "second-half impressions/day < 0.8 * first-half impressions/day",
    "split": "GroupShuffleSplit by client_hash_id",
    "random_state": 42,
    "excluded": ["client_hash_id", "content_hash_id", "second-half outcomes", "label-derived fields", "product flags"],
}
print("Features:", method_facts["features"])
print("Label:", method_facts["label_proxy"])
print("Validation:", method_facts["split"], "random_state=42")
assert len(feature_columns) == 5
assert "client_hash_id" in method_facts["excluded"]

Features: ['first_half_impressions', 'first_half_clicks', 'first_half_ctr', 'first_half_avg_position', 'first_half_active_days']
Label: second-half impressions/day < 0.8 * first-half impressions/day
Validation: GroupShuffleSplit by client_hash_id random_state=42


## 4. Results versus baseline

The Random Forest is compared with the frozen ML-07 rule on the same 11 held-out clients. Precision@K is the lane metric because the output is a ranked review queue. The model improves the top-of-queue precision, while the base rate and error counts keep the result in decision-support territory.

In [11]:
results = pd.DataFrame([
    {"method": "ML-07 transparent baseline", "metric": "precision@50", "value": 0.22},
    {"method": "ML-08 Random Forest", "metric": "precision@50", "value": 0.44},
    {"method": "ML-07 transparent baseline", "metric": "precision@100", "value": 0.28},
    {"method": "ML-08 Random Forest", "metric": "precision@100", "value": 0.43},
])
print(results.to_string(index=False))
print("Held-out clients: 11")
print("Held-out decline-proxy base rate: 0.393")
print("Held-out ROC AUC: 0.573")
print("Error counts: 8,567 correct; 5,205 false negatives; 748 false positives")
assert results.loc[(results["method"] == "ML-08 Random Forest") & (results["metric"] == "precision@50"), "value"].iloc[0] > results.loc[(results["method"] == "ML-07 transparent baseline") & (results["metric"] == "precision@50"), "value"].iloc[0]
assert results.loc[(results["method"] == "ML-08 Random Forest") & (results["metric"] == "precision@100"), "value"].iloc[0] > results.loc[(results["method"] == "ML-07 transparent baseline") & (results["metric"] == "precision@100"), "value"].iloc[0]

                    method        metric  value
ML-07 transparent baseline  precision@50   0.22
       ML-08 Random Forest  precision@50   0.44
ML-07 transparent baseline precision@100   0.28
       ML-08 Random Forest precision@100   0.43
Held-out clients: 11
Held-out decline-proxy base rate: 0.393
Held-out ROC AUC: 0.573
Error counts: 8,567 correct; 5,205 false negatives; 748 false positives


## 5. Limitations

This is one mid-panel month from an unbalanced panel. The results measure observed associations and review opportunity; they do not prove that changing a title, snippet, or page will improve rankings. The model improves top-of-queue precision on held-out clients, but its ROC AUC is 0.573 and it produces many false negatives. Query mix, client history, missingness, and future-month stability still need deeper analysis. The project does not predict Google's algorithm.

In [12]:
limitations = [
    "single March 2026 mid-panel slice",
    "unbalanced client history",
    "observational data without a causal intervention",
    "model ROC AUC is 0.573 and false negatives outnumber false positives",
    "future-month stability and richer query-mix signals remain untested",
]
for limitation in limitations:
    print("-", limitation)
assert len(limitations) == 5

- single March 2026 mid-panel slice
- unbalanced client history
- observational data without a causal intervention
- model ROC AUC is 0.573 and false negatives outnumber false positives
- future-month stability and richer query-mix signals remain untested


## 6. Ranked recommendations

The queue supports human review in this order: first inspect high-volume, low-CTR candidates; then check query mix and title/snippet alignment; monitor low-evidence pages rather than forcing an intervention; and use the Random Forest as a second ranking aid only after checking its held-out performance and error profile.

In [15]:
recommendations = pd.DataFrame([
    {"rank": 1, "action": "Review high-volume, low-CTR pages", "reason": "measurable opportunity plus position-aware CTR concern", "confidence": "directional"},
    {"rank": 2, "action": "Check query mix and snippet alignment", "reason": "the model and rule cannot distinguish intent mismatch from fixable copy", "confidence": "low to moderate"},
    {"rank": 3, "action": "Monitor low-evidence pages", "reason": "sparse history does not support a strong intervention", "confidence": "low"},
    {"rank": 4, "action": "Use the Random Forest as a second ranking aid", "reason": "it improves held-out precision but still has many false negatives", "confidence": "moderate for prioritization"},
])
print(recommendations.to_string(index=False))
assert recommendations["rank"].tolist() == [1, 2, 3, 4]

 rank                                        action                                                                  reason                  confidence
    1             Review high-volume, low-CTR pages                  measurable opportunity plus position-aware CTR concern                 directional
    2         Check query mix and snippet alignment the model and rule cannot distinguish intent mismatch from fixable copy             low to moderate
    3                    Monitor low-evidence pages                   sparse history does not support a strong intervention                         low
    4 Use the Random Forest as a second ranking aid       it improves held-out precision but still has many false negatives moderate for prioritization


## 7. Artifacts and reproducibility

The completed artifacts are the ML-02 through ML-08 notebooks and the report at `work/capstone_report.md`. ML-07 regenerates `work/outputs/baseline_action_score.csv`; ML-08 writes `work/outputs/model_metrics.json`. The CSV is intentionally ignored by Git. The notebooks use DuckDB against the March warehouse partition, a hidden Hugging Face read-token prompt, and `random_state=42` for validation.

In [14]:
artifacts = {
    "report": "work/capstone_report.md",
    "contract_notebook": "work/notebooks/w03_data_contract.ipynb",
    "leakage_notebook": "work/notebooks/w03_feature_leakage_check.ipynb",
    "baseline_notebook": "work/notebooks/w04_baseline_score.ipynb",
    "model_notebook": "work/notebooks/w05_model.ipynb",
    "generated_queue": "work/outputs/baseline_action_score.csv",
    "model_metrics": "work/outputs/model_metrics.json",
    "data_safety": "no raw queries, URLs, client names, or tokens committed",
}
for name, path in artifacts.items():
    print(f"{name}: {path}")
assert artifacts["data_safety"].startswith("no raw")

report: work/capstone_report.md
contract_notebook: work/notebooks/w03_data_contract.ipynb
leakage_notebook: work/notebooks/w03_feature_leakage_check.ipynb
baseline_notebook: work/notebooks/w04_baseline_score.ipynb
model_notebook: work/notebooks/w05_model.ipynb
generated_queue: work/outputs/baseline_action_score.csv
model_metrics: work/outputs/model_metrics.json
data_safety: no raw queries, URLs, client names, or tokens committed


## Self-check

- [x] Ranking Signal Analysis question and decision are stated.
- [x] March warehouse data, windows, exclusions, and safety rules are documented.
- [x] Five decision-time features and the observed decline proxy are defined.
- [x] ML-07 baseline and ML-08 model results are compared on the same grouped test split.
- [x] Limitations and ranked recommendations are included.
- [x] Report, notebook, queue, and metrics artifacts are named.
- [x] All capstone code cells execute successfully.